# CDI Layer 2 and Layer 3

Use the **compute** kernel. After repository code changes, restart the kernel once. Put `OPENAI_API_KEY` in the repository `.env`, set the fact-sheet path and Layer 2 reasoning effort below, and run this cell. It creates a resumable eight-mission Layer 2 run and shows the recorded usage plus one mission for inspection.

In [ ]:
from __future__ import annotations

import asyncio
import os
from pathlib import Path

from IPython.display import JSON, Markdown, display

from ML.deep_research.layer2.cli import load_dotenv_key, run_all as run_layer2
from ML.deep_research.layer2.create_run import create_run as create_layer2_run
from ML.deep_research.layer2.fs import load_json
from ML.deep_research.layer2.settings import PLANNER_PATH, RUNS_DIR

FACT_SHEET_PATH = Path(r"inputs\1_Amtsgericht_Bad-Homburg_e0b0ca5b_fact_sheet.md")
LAYER2_REASONING_EFFORT = "medium"  # low | medium | high | max
PREVIEW_MISSION = 0

load_dotenv_key()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to the repository .env before running Layer 2.")

L2_RUN = create_layer2_run(
    FACT_SHEET_PATH,
    PLANNER_PATH,
    RUNS_DIR,
    reasoning_effort=LAYER2_REASONING_EFFORT,
)
print(f"Layer 2 run created: {L2_RUN}")
print(f"Resume if interrupted: .\\run.ps1 -Resume '{L2_RUN}'")
L2_USAGE = await asyncio.to_thread(run_layer2, L2_RUN)

mission_files = sorted((L2_RUN / "missions").glob("*.json"))
display(Markdown("### Recorded Layer 2 usage"))
display(JSON(data=L2_USAGE, expanded=True))
print(f"Mission files: {len(mission_files)}")
for path in mission_files:
    print(f"- {path.name}")
if mission_files:
    preview = mission_files[PREVIEW_MISSION]
    display(Markdown(f"### Mission preview: `{preview.name}`"))
    display(JSON(data=load_json(preview), expanded=False))


## Layer 3 — live online research

Run this only after Layer 2 completes. **This cell confirms that the input is public or invented, sends research queries to external services, and consumes model/web-search usage.** Set the exact Layer 2 run path, Layer 3 model reasoning, web-search depth, and web-search verbosity below. A blank run path uses `L2_RUN` from the Layer 2 cell.

In [1]:
from __future__ import annotations

import asyncio
import json
import os
from pathlib import Path

from IPython.display import JSON, Markdown, clear_output, display

from ML.deep_research.layer2.cli import load_dotenv_key
from ML.deep_research.layer2.fs import load_json, read_text
from ML.deep_research.layer3.cli import run_all as run_layer3
from ML.deep_research.layer3.pipeline.create_run import create_run as create_layer3_run
from ML.deep_research.layer3.settings import RUNS_DIR as LAYER3_RUNS_DIR, SCHEMA_VERSION
from ML.deep_research.layer3.usage import summarize_usage

LAYER3_SOURCE_RUN_PATH = r"runs\inputs-1-amtsgericht-bad-homburg-e0b0ca5b-fact-sheet-0d62dba4\L2_20260821_113445_4a51"  # paste a runs/.../L2_* folder; blank uses L2_RUN above
LAYER3_MODEL_REASONING_EFFORT = "low"  # low | medium | high | max
WEB_SEARCH_DEPTH = "low"  # low | medium | high
WEB_SEARCH_VERBOSITY = "low"  # low | medium | high
PUBLIC_INPUT_CONFIRMED = True

load_dotenv_key()
if LAYER3_SOURCE_RUN_PATH.strip():
    L2_RUN = Path(LAYER3_SOURCE_RUN_PATH)
elif globals().get("L2_RUN"):
    L2_RUN = Path(L2_RUN)
else:
    raise RuntimeError("Set LAYER3_SOURCE_RUN_PATH to the exact Layer 2 run folder.")
print(f"Using existing Layer 2 run: {L2_RUN}")

if not PUBLIC_INPUT_CONFIRMED:
    raise RuntimeError("Layer 3 requires explicit confirmation of public or invented input.")
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to the repository .env before running Layer 3.")

for candidate in sorted(
    LAYER3_RUNS_DIR.rglob("L3_*"), key=lambda path: path.stat().st_mtime, reverse=True
):
    candidate_record = load_json(candidate / "run.json")
    source_path = candidate_record.get("source_l2", {}).get("path", "")
    search_options = candidate_record.get("web_search", {})
    if (
        candidate_record.get("schema_version") == SCHEMA_VERSION
        and source_path
        and Path(source_path).resolve() == L2_RUN.resolve()
        and candidate_record.get("reasoning_effort") == LAYER3_MODEL_REASONING_EFFORT
        and search_options.get("context_size") == WEB_SEARCH_DEPTH
        and search_options.get("verbosity") == WEB_SEARCH_VERBOSITY
    ):
        L3_RUN = candidate
        break
else:
    L3_RUN = create_layer3_run(
        L2_RUN,
        LAYER3_RUNS_DIR,
        public_input_confirmed=PUBLIC_INPUT_CONFIRMED,
        reasoning_effort=LAYER3_MODEL_REASONING_EFFORT,
        web_search_context_size=WEB_SEARCH_DEPTH,
        web_search_verbosity=WEB_SEARCH_VERBOSITY,
    )

l3_before = load_json(L3_RUN / "run.json")
execution = l3_before.get("execution", {})
resume_command = f".\\run.ps1 -ResumeL3 '{L3_RUN}'"
print(f"Layer 3 run: {L3_RUN}")
print(f"Resume if interrupted: {resume_command}")

layer3_task = None
if l3_before.get("status") != "complete":
    layer3_task = asyncio.create_task(run_layer3(L3_RUN))
while layer3_task and not layer3_task.done():
    await asyncio.sleep(5)
    live = load_json(L3_RUN / "run.json")
    domains = live.get("execution", {}).get("domains", {})
    running = [name for name, item in domains.items() if item.get("status") == "running"]
    completed = [name for name, item in domains.items() if item.get("status") == "complete"]
    domain_finals = sorted((L3_RUN / "domains").glob("*/final.md"))
    lines = read_text(L3_RUN / "usage.jsonl").splitlines()
    try:
        latest = json.loads(lines[-1]) if lines else {}
    except json.JSONDecodeError:
        latest = {}
    clear_output(wait=True)
    print(f"Layer 3 run: {L3_RUN}")
    print(f"Resume if interrupted: {resume_command}")
    print(f"Active coordinator: {running[0] if running else 'none'}")
    print(f"Completed coordinators: {len(completed)}/8")
    print(f"Saved domain responses: {len(domain_finals)}/8")
    print("Usage:", summarize_usage(L3_RUN))
    if latest:
        print("Last activity:", latest.get("timestamp"), latest.get("actor"), latest.get("phase"), latest.get("detail", ""))

L3_ERROR = ""
if layer3_task:
    try:
        await layer3_task
    except Exception as error:
        L3_ERROR = f"{type(error).__name__}: {error}"
clear_output(wait=True)

l3_record = load_json(L3_RUN / "run.json")
domain_finals = sorted((L3_RUN / "domains").glob("*/final.md"))
final_answer = L3_RUN / "research" / "final.md"
final_records = [*l3_record.get("execution", {}).get("domains", {}).values(), l3_record.get("execution", {}).get("final", {})]
print(f"Layer 3 run: {L3_RUN}")
print(f"Status: {l3_record.get('status', 'unknown')}")
print(f"Resume if interrupted: .\\run.ps1 -ResumeL3 '{L3_RUN}'")
if L3_ERROR:
    print(f"Execution error: {L3_ERROR}")
failed = [
    f"{item.get('stage')}: {item.get('actor')} — {item.get('error')}"
    for item in final_records
    if item.get("status") == "failed"
]
for item in failed:
    print(f"Failed stage: {item}")
display(Markdown("### Recorded Layer 3 usage"))
display(JSON(data=summarize_usage(L3_RUN), expanded=True))
print(f"Saved domain responses: {len(domain_finals)}/8")
for path in domain_finals:
    print(f"- {path.relative_to(L3_RUN)}")
display(Markdown("### Property synthesis"))
display(Markdown(read_text(final_answer) if final_answer.is_file() else "Not produced; resume the incomplete Layer 3 run shown above."))


Layer 3 run: C:\Users\Arjun Gowda\Desktop\Sed_ai_v2\runs\inputs-1-amtsgericht-bad-homburg-e0b0ca5b-fact-sheet-0d62dba4\L3_20260824_111800_6370
Status: incomplete
Resume if interrupted: .\run.ps1 -ResumeL3 'C:\Users\Arjun Gowda\Desktop\Sed_ai_v2\runs\inputs-1-amtsgericht-bad-homburg-e0b0ca5b-fact-sheet-0d62dba4\L3_20260824_111800_6370'
Failed stage: coordinator: Rights, Public Law & Ownership Governance — RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
Failed stage: coordinator: Ground, Physical Climate & Insurability — RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credi

### Recorded Layer 3 usage

<IPython.core.display.JSON object>

Saved domain responses: 2/8
- domains\asset-integrity-systems-and-operational-resilience\final.md
- domains\occupier-lease-income-and-counterparty-economics\final.md


### Property synthesis

Not produced; resume the incomplete Layer 3 run shown above.